# 06 - Large-Scale Vectorized Fleet Aggregation (10,000+ DERs)

Simulating thousands of individual Distributed Energy Resources (DERs) in Python typically causes severe performance degradation due to object dispatch overhead.

In this notebook, we demonstrate **Vectorized Fleet Containers**:
1. Simulating 10,000 heterogeneous residential batteries.
2. Simulating 5,000 rooftop solar PV arrays.
3. Microsecond execution benchmarks across varying fleet sizes.


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from energysim import world

print("Vectorized container benchmark ready!")


## Step 1: Add a 10,000-Unit Residential Battery Fleet


In [ ]:
N_BATTERIES = 10000
np.random.seed(42)

capacities = np.random.uniform(10.0, 20.0, N_BATTERIES)   # 10 to 20 kWh
initial_socs = np.random.uniform(0.2, 0.8, N_BATTERIES)

sim = world(start_time=0, stop_time=86400, t_macro=900)

sim.add_container(
    sim_name="home_storage",
    container_type="battery_fleet",
    size=N_BATTERIES,
    capacity_kwh=capacities,
    max_charge_kw=5.0,
    max_discharge_kw=5.0,
    initial_soc=initial_socs,
    outputs=["total_actual_p_kw", "mean_soc", "min_soc", "max_soc"]
)

# Broadcast solar absorption command
sim.add_signal("dr_command", lambda t: 3.5 if 36000 <= t <= 50400 else -2.0)
sim.add_connections({"dr_command.y": "home_storage.p_cmd_kw"})

t0 = time.time()
sim.simulate(dashboard=False)
elapsed = time.time() - t0

print(f"Successfully simulated {N_BATTERIES:,} batteries over 24 hours in {elapsed:.2f} seconds!")


## Step 2: Inspect Fleet Results


In [ ]:
df_fleet = sim.results(dashboard=False)["home_storage"]
display(df_fleet.head())
